## Primary Formal Analysis

BGE is not included in this model; it remains a secondary analysis. Here, we combine only the 110 confirmatory results $N=50/200$ with the 75 historical anchors $N=100$, while retaining a `source` column to clearly indicate their origin.

In [1]:
from pathlib import Path
import numpy as np,pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import chi2

ROOT=Path("..")
A=ROOT/"results"/"week12"/"confirmatory"/"analysis"
OUT=A/"formal_model"
OUT.mkdir(exist_ok=True)

confirm=pd.read_csv(A/"power_wilson.csv")
print(confirm.shape)

(110, 13)


In [2]:
# frozen historical N=100 anchors
SEV=[.1,.3,.6]

anchors={
"Broken Timing":{
"C2ST-Linear":[.025,.835,1],"C2ST-RBF":[.040,.820,1],
"Energy":[.975,1,1],"MMD":[.975,1,1],"Sinkhorn":[.330,.940,1]},
"Entity Inconsistency":{
"C2ST-Linear":[0,.025,.165],"C2ST-RBF":[0,.015,.180],
"Energy":[.605,1,1],"MMD":[.640,1,1],"Sinkhorn":[.045,.025,.090]},
"Missing Cross-Channel Reference":{
"C2ST-Linear":[.005,.125,.505],"C2ST-RBF":[0,.080,.515],
"Energy":[.140,.550,.995],"MMD":[.140,.560,.995],"Sinkhorn":[.125,.310,.685]},
"Over-Polished Language":{
"C2ST-Linear":[0,.280,.930],"C2ST-RBF":[0,.185,.725],
"Energy":[.870,1,1],"MMD":[.880,1,1],"Sinkhorn":[.130,.025,.040]},
"Template Repetition":{
"C2ST-Linear":[.330,1,1],"C2ST-RBF":[.490,1,1],
"Energy":[.950,1,1],"MMD":[.950,1,1],"Sinkhorn":[.970,1,1]}
}

rows=[
    {"scenario":s,"severity":z,"test":t,"power":p}
    for s,d in anchors.items()
    for t,v in d.items()
    for z,p in zip(SEV,v)
]

a100=pd.DataFrame(rows)
a100["N"]=100
a100["n_reps"]=200
a100["rejections"]=(a100.power*200).round().astype(int)
a100["source"]="historical_n100_anchor"

assert len(a100)==75
a100.head()

,scenario,severity,test,power,N,n_reps,rejections,source
0,Broken Timing,0.1,C2ST-Linear,0.025,100,200,5,historical_n100_anchor
1,Broken Timing,0.3,C2ST-Linear,0.835,100,200,167,historical_n100_anchor
2,Broken Timing,0.6,C2ST-Linear,1.000,100,200,200,historical_n100_anchor
3,Broken Timing,0.1,C2ST-RBF,0.040,100,200,8,historical_n100_anchor
4,Broken Timing,0.3,C2ST-RBF,0.820,100,200,164,historical_n100_anchor


In [3]:
# Combine 50/100/200
new=confirm[
    ["scenario","severity","N","test","rejections","n_reps","power"]
].copy()

new["source"]="confirmatory_n50_n200"

df=pd.concat([new,a100],ignore_index=True)

assert len(new)==110
assert len(df)==185
assert df.duplicated(["scenario","severity","N","test"]).sum()==0
assert (df.n_reps==200).all()

print(df.groupby(["source","N"]).size())
print("Total cells:",len(df))

df.to_csv(OUT/"primary_model_dataset.csv",index=False)

source                  N  
confirmatory_n50_n200   50     60
                        200    50
historical_n100_anchor  100    75
dtype: int64
Total cells: 185


In [4]:
# Freeze categorical reference levels
SCENARIOS=[
    "Broken Timing","Entity Inconsistency",
    "Missing Cross-Channel Reference",
    "Over-Polished Language","Template Repetition"
]
TESTS=["C2ST-Linear","C2ST-RBF","Energy","MMD","Sinkhorn"]

df["scenario"]=pd.Categorical(df.scenario,categories=SCENARIOS)
df["severity"]=pd.Categorical(df.severity,categories=[.1,.3,.6])
df["N"]=pd.Categorical(df.N,categories=[100,50,200])
df["test"]=pd.Categorical(df.test,categories=TESTS)

df["power"]=df.rejections/df.n_reps

In [5]:
# Preregistered binomial logistic model
MAIN="C(scenario)+C(severity)+C(N)+C(test)"

INT={
    "Scenario × Test":"C(scenario):C(test)",
    "Severity × Test":"C(severity):C(test)",
    "N × Test":"C(N):C(test)"
}

FORMULA="power ~ "+MAIN+"+"+"+".join(INT.values())

model=smf.glm(
    FORMULA,
    data=df,
    family=sm.families.Binomial(),
    var_weights=df.n_reps
).fit()

print("Converged:",model.converged)
print("Cells:",len(df))
print("Parameters:",len(model.params))
print("Deviance:",round(model.deviance,3))
print("AIC:",round(model.aic,3))

Converged: True
Cells: 185
Parameters: 45
Deviance: 1646.211
AIC: 14498.635


In [6]:
# Design matrix integrity
X=model.model.exog
rank=np.linalg.matrix_rank(X)

print("Design matrix:",X.shape)
print("Rank:",rank)
print("Full rank:",rank==X.shape[1])

assert model.converged
assert rank==X.shape[1]

Design matrix: (185, 45)
Rank: 45
Full rank: True


In [7]:
# Omnibus interaction tests
lrt=[]

for name,term in INT.items():
    reduced_terms=[v for k,v in INT.items() if k!=name]
    f="power ~ "+MAIN+"+"+"+".join(reduced_terms)

    reduced=smf.glm(
        f,data=df,
        family=sm.families.Binomial(),
        var_weights=df.n_reps
    ).fit()

    LR=2*(model.llf-reduced.llf)
    dof=int(model.df_model-reduced.df_model)

    lrt.append({
        "interaction":name,
        "LR":LR,
        "df":dof,
        "p_value":chi2.sf(LR,dof)
    })

lrt=pd.DataFrame(lrt)
lrt

,interaction,LR,df,p_value
0,Scenario × Test,5014.185529,16,0.000000e+00
1,Severity × Test,1046.823757,8,1.164074e-220
2,N × Test,173.295525,8,2.627829e-33


In [8]:
# N × Test only
lrt[lrt.interaction=="N × Test"]

,interaction,LR,df,p_value
2,N × Test,173.295525,8,2.627829e-33


In [9]:
# Coefficients + odds ratios
ci=model.conf_int()

coef=pd.DataFrame({
    "term":model.params.index,
    "beta":model.params.values,
    "SE":model.bse.values,
    "p_value":model.pvalues.values,
    "OR":np.exp(model.params.values),
    "OR_low":np.exp(ci[0].values),
    "OR_high":np.exp(ci[1].values)
})

coef.to_csv(OUT/"primary_glm_coefficients.csv",index=False)
coef.round(4)

,term,beta,SE,p_value,OR,OR_low,OR_high
0,Intercept,-3.0570,0.1350,0.0000,0.0470,0.0361,0.0613
1,C(scenario)[T.Entity Inconsistency],-5.7008,0.1736,0.0000,0.0033,0.0024,0.0047
2,C(scenario)[T.Missing Cross-Channel Reference],-4.2523,0.1479,0.0000,0.0142,0.0107,0.0190
3,C(scenario)[T.Over-Polished Language],-2.4135,0.1268,0.0000,0.0895,0.0698,0.1147
4,C(scenario)[T.Template Repetition],2.6727,0.1404,0.0000,14.4796,10.9957,19.0673
5,C(severity)[T.0.3],4.5631,0.1449,0.0000,95.8767,72.1664,127.3770
6,C(severity)[T.0.6],7.4319,0.1879,0.0000,1689.0731,1168.6791,2441.1901
7,C(N)[T.50],-2.3242,0.1111,0.0000,0.0979,0.0787,0.1217
8,C(N)[T.200],2.4006,0.1020,0.0000,11.0294,9.0313,13.4697
9,C(test)[T.C2ST-RBF],-0.0521,0.1939,0.7881,0.9492,0.6492,1.3880


In [10]:
# Only N × Test coefficients
n_test=coef[
    coef.term.str.contains(r"C\(N\)",regex=True) &
    coef.term.str.contains(":",regex=False)
].copy()

n_test.round(4) 


,term,beta,SE,p_value,OR,OR_low,OR_high
37,C(N)[T.50]:C(test)[T.C2ST-RBF],-0.1197,0.1597,0.4537,0.8872,0.6488,1.2133
38,C(N)[T.200]:C(test)[T.C2ST-RBF],0.1031,0.1466,0.4818,1.1086,0.8317,1.4777
39,C(N)[T.50]:C(test)[T.Energy],-0.3760,0.1583,0.0175,0.6866,0.5035,0.9364
40,C(N)[T.200]:C(test)[T.Energy],0.9529,0.2194,0.0000,2.5931,1.6869,3.9862
41,C(N)[T.50]:C(test)[T.MMD],-0.4385,0.1592,0.0059,0.6450,0.4721,0.8811
42,C(N)[T.200]:C(test)[T.MMD],0.8742,0.2194,0.0001,2.3971,1.5594,3.6847
43,C(N)[T.50]:C(test)[T.Sinkhorn],0.1585,0.1485,0.2858,1.1717,0.8759,1.5675
44,C(N)[T.200]:C(test)[T.Sinkhorn],-0.8182,0.1338,0.0000,0.4412,0.3395,0.5735


In [11]:
# Adjusted N × Test predictions
grid=pd.MultiIndex.from_product(
    [SCENARIOS,[.1,.3,.6],[100,50,200],TESTS],
    names=["scenario","severity","N","test"]
).to_frame(index=False)

grid["predicted_power"]=model.predict(grid)

adjusted=(
    grid.groupby(["N","test"],as_index=False)
    .predicted_power.mean()
)

adjusted.pivot(
    index="test",
    columns="N",
    values="predicted_power"
).round(3)

N,50,100,200
test,,,
C2ST-Linear,0.246,0.415,0.619
C2ST-RBF,0.232,0.403,0.611
Energy,0.662,0.872,0.969
MMD,0.663,0.876,0.969
Sinkhorn,0.243,0.448,0.619


In [12]:
# Change relative to N=100
wide=adjusted.pivot(
    index="test",columns="N",values="predicted_power"
)

wide["Δ50_vs_100"]=wide[50]-wide[100]
wide["Δ200_vs_100"]=wide[200]-wide[100]

wide.round(3)

N,50,100,200,Δ50_vs_100,Δ200_vs_100
test,,,,,
C2ST-Linear,0.246,0.415,0.619,-0.169,0.204
C2ST-RBF,0.232,0.403,0.611,-0.171,0.208
Energy,0.662,0.872,0.969,-0.210,0.097
MMD,0.663,0.876,0.969,-0.213,0.093
Sinkhorn,0.243,0.448,0.619,-0.205,0.171


In [13]:
# save outputs 
lrt.to_csv(OUT/"primary_interaction_LRT.csv",index=False)
n_test.to_csv(OUT/"N_by_test_coefficients.csv",index=False)
adjusted.to_csv(OUT/"adjusted_N_by_test_power.csv",index=False)

with open(OUT/"primary_glm_summary.txt","w") as f:
    f.write(model.summary().as_text())

print("Saved:",OUT.resolve())

Saved: /Users/jessica/fidelity-metric-lab/results/week12/confirmatory/analysis/formal_model


In [14]:
lrt.round(4)

,interaction,LR,df,p_value
0,Scenario × Test,5014.1855,16,0.0
1,Severity × Test,1046.8238,8,0.0
2,N × Test,173.2955,8,0.0


In [15]:
n_test.round(4)

,term,beta,SE,p_value,OR,OR_low,OR_high
37,C(N)[T.50]:C(test)[T.C2ST-RBF],-0.1197,0.1597,0.4537,0.8872,0.6488,1.2133
38,C(N)[T.200]:C(test)[T.C2ST-RBF],0.1031,0.1466,0.4818,1.1086,0.8317,1.4777
39,C(N)[T.50]:C(test)[T.Energy],-0.3760,0.1583,0.0175,0.6866,0.5035,0.9364
40,C(N)[T.200]:C(test)[T.Energy],0.9529,0.2194,0.0000,2.5931,1.6869,3.9862
41,C(N)[T.50]:C(test)[T.MMD],-0.4385,0.1592,0.0059,0.6450,0.4721,0.8811
42,C(N)[T.200]:C(test)[T.MMD],0.8742,0.2194,0.0001,2.3971,1.5594,3.6847
43,C(N)[T.50]:C(test)[T.Sinkhorn],0.1585,0.1485,0.2858,1.1717,0.8759,1.5675
44,C(N)[T.200]:C(test)[T.Sinkhorn],-0.8182,0.1338,0.0000,0.4412,0.3395,0.5735


In [16]:
wide.round(3)

N,50,100,200,Δ50_vs_100,Δ200_vs_100
test,,,,,
C2ST-Linear,0.246,0.415,0.619,-0.169,0.204
C2ST-RBF,0.232,0.403,0.611,-0.171,0.208
Energy,0.662,0.872,0.969,-0.210,0.097
MMD,0.663,0.876,0.969,-0.213,0.093
Sinkhorn,0.243,0.448,0.619,-0.205,0.171


The effect of sample size depends on the test configuration, but this effect is also heavily conditioned by the defect and severity.

The key result is:
`N × Test`
$$LR = 173.30$$
$$df = 8$$
$$p < .001$$

**Important clarifications:**
The $$0.0$$ value does not mean the probability is zero, it is an effect of rounding/numerical precision. In the paper it will be reported as $$p < .001$$.

The following are also significant:
`Scenario × Test` $$LR = 5014.19$$, $$df = 16$$, $$p < .001$$
`Severity × Test` $$LR = 1046.82$$, $$df = 8$$, $$p < .001$$

This fits perfectly with the central idea of the project: there is no metric that is simply "good" or "bad"; its response changes dramatically depending on the type of defect.


### What `N × Test` specifically indicates
The interaction confirms that increasing or decreasing $$N$$ does not affect the five configurations equally. 

The adjusted table shows this intuitively:

| Test | N=50 | N=100 | N=200 |
| :--- | :---: | :---: | :---: |
| `C2ST-Linear` | $$.246$$ | $$.415$$ | $$.619$$ |
| `C2ST-RBF` | $$.232$$ | $$.403$$ | $$.611$$ |
| `Energy` | $$.662$$ | $$.872$$ | $$.969$$ |
| `MMD` | $$.663$$ | $$.876$$ | $$.969$$ |
| `Sinkhorn` | $$.243$$ | $$.448$$ | $$.619$$ |

`Energy` and `MMD` remain far more powerful in absolute terms across all three sample sizes. Even at $$N=50$$, they have an adjusted power of around $$.66$$, while `C2ST` and `Sinkhorn` sit around $$0.23–0.25$$.

However, there is an important detail when interpreting the individual coefficients.

For $$N=50$$:
* `Energy × N50` $$\beta = -0.376$$, $$p = .0175$$
* `MMD × N50` $$\beta = -0.439$$, $$p = .0059$$

This means that, on a log-odds scale, reducing from $$N=100$$ to $$N=50$$ affects `Energy` and `MMD` differently than `C2ST-Linear`.

And for $$N=200$$:
* `Energy × N200` $\beta = .953$, $p < .001$$
* `MMD × N200` $\beta = .874$, $$p < .001$$


There is also a significant difference compared to the pattern of `C2ST-Linear`. This doesn't mean `C2ST` increased more.
This happens because `Energy` was already at $$0.872$$ with $$N=100$$, leaving very little room to grow before hitting the ceiling of $1.0$. The regression works in log-odds, not directly in absolute power differences. This is a ceiling effect.


### C2ST Linear vs RBF

* `N50 × C2ST-RBF` $$p = 0.454$$
* `N200 × C2ST-RBF` $$p = 0.482$$

There was no evidence that the sample size effect is substantially different between `C2ST-Linear` and `C2ST-RBF`.
This complements what was already observed descriptively: 
> *"After aligning the significance procedure, the RBF comparison suggests that classifier linearity alone does not explain the main C2ST blind spots under the tested representations."*


### Sinkhorn

`N50 × Sinkhorn`:
$$\beta = .159$$
$$p = .286$$

There is no clear evidence that dropping to $$N=50$$ affects `Sinkhorn` differently than `C2ST-Linear`.

However:
`N200 × Sinkhorn`:
$$\beta = -.818$$
$$OR = .441$$
$$p < .001$$

This does indicate that its response to increasing to $$N=200$$ is significantly different from that of `C2ST-Linear` on a log-odds scale.vThis aligns with what the observed scenario by scenario: `Sinkhorn` has a much more irregular response that is highly dependent on representation/defect compared to `Energy` and `MMD`.


### Most Important Model Finding


> *"The preregistered binomial model showed a significant Sample Size × Test Configuration interaction (LR $$\chi^2(8) = 173.30$$, $$p < .001$$), indicating that changes in sample size affected the five test configurations differently. Significant Scenario × Test Configuration and Severity × Test Configuration interactions further showed that detection performance depended strongly on the type and prevalence of the injected defect."*

> *"MMD and Energy Distance maintained the highest standardized detection power across sample sizes, whereas C2ST and Sinkhorn showed substantially stronger defect-specific limitations."*

This does not mean that the model proves `Energy`/`MMD` are "universally better." `Scenario × Test` specifically tells that superiority depends on the scenario.


### Key finding that can be now asserted 
The conceptual hypothesis:

$$\text{Fidelity response} = \text{Defect} \times \text{Representation} \times \text{Metric/Test Configuration} \times \text{Test design}$$

is now supported from multiple angles:
* `Scenario × Test`: The defect changes which test works.
* `Severity × Test`: Prevalence alters the response of each test differently.
* `N × Test`: Sample size alters the response of each test differently.
* `BGE`: Changing the representation modifies some blind spots of `C2ST` and `Sinkhorn`.
* `Energy/MMD`: Maintain an extraordinary similarity across many conditions.
